# Reranker
Use this notebook to configure your retriever pipelines and test them on `train.jsonl` as validation.

Current Reranker Selection:
- [] Qwen/qwen2.5-7b-instruct
- [] Qwen/qwen2.5-32b-instruct

In [ ]:
# === IMPORT LIBRARIES === 
import sys
import os
from pathlib import Path
import ctypes
import pandas as pd

In [ ]:
# === RESOLVE PATHS ===

# CRITICAL SYSTEM BOOT FIX: Force-inject CUDA library to RAM before ANY module imports!
try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
except Exception:
    pass

# SYSTEM HOTFIX: Inject absolute path to CUDA 13.0 linker libraries
# This guarantees that bitsandbytes and 4-bit quantization load flawlessly on this server!
cuda_link_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib"
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + cuda_link_path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Add parent directory to python path to import from src/
sys.path.append(str(Path("..").resolve()))

In [ ]:
# === IMPORT MODULES ===
from src.data.loader import load_jsonl
from src.evaluation.metrics import evaluate_retriever
from src.retrievers.bm25 import BM25Retriever
from src.retrievers.dense import DenseRetriever
from src.retrievers.llm import LLMRetriever
from src.retrievers.cross_encoder import CrossEncoderRetriever
from src.retrievers.cached import CachedRetriever
from src.evaluation import calculate_calibration_metrics
from src.evaluation.naming import generate_experiment_name

# === IMPORT HF API TOKEN ===
env_path = Path("..") / ".env"
if env_path.exists():
    with open(env_path, "r") as f:
        for line in f:
            if "=" in line and not line.strip().startswith("#"):
                k, v = line.strip().split("=", 1)
                os.environ[k.strip()] = v.strip()
    print("✅ Successfully Authenticated Hugging Face Token!")


In [ ]:
# === IMPORT VALIDATION DATA ===
DATA_DIR = Path("../data")
val_data = load_jsonl(str(DATA_DIR / "train.jsonl"))
print(f"Loaded {len(val_data)} validation samples.")

In [ ]:
# === EXPERIMENT CONFIGURATION ===
CONFIG = {
    "retriever": "bm25", 
    "dense_model_name": "BAAI/bge-large-en-v1.5", 
    "llm_model_name": "Qwen/Qwen2.5-32B-Instruct",
    "dense_scores_path": None,
    "top_k": 5,
    "max_candidates": 12,
    "max_chars": None
}

# To run none_hybrid(qwen3-embedding-8b + bge-m3)_none_none
CONFIG = {
    "retriever": "cached",  # <-- Change this to cached
    "dense_scores_path": "../outputs/cache/ensemble_dense_scores/qwen3-embedding-8b_w0.6_bge-m3_w0.4_dense_scores_train.json",
    "top_k": 5,
}

os.environ["DISABLE_VLM"] = "1" # 0: no, 1: yes
os.environ["VLM_CAPTIONS_FILE"] = "vlm_image_captions/qwen2-vl-7b-instruct_image_captions.json"

In [ ]:
# === RETRIEVER SELECTION ===
if CONFIG["retriever"] == "bm25":
    retriever = BM25Retriever()
elif CONFIG["retriever"] == "dense":
    retriever = DenseRetriever(
        model_name=CONFIG["dense_model_name"]
    )
elif CONFIG["retriever"] == "llm":
    retriever = LLMRetriever(
        model_name=CONFIG["llm_model_name"], 
        load_in_4bit=True, 
        disable_filtering=False,
        dense_scores_path=CONFIG["dense_scores_path"],
        max_candidates=CONFIG.get("max_candidates", 12),
        max_chars=CONFIG.get("max_chars", None)
    )
elif CONFIG["retriever"] == "cross_encoder":
    retriever = CrossEncoderRetriever(
        model_name=CONFIG["llm_model_name"],
        load_in_4bit=True,
        disable_filtering=False,
        dense_scores_path=CONFIG["dense_scores_path"],
        max_candidates=CONFIG.get("max_candidates", 12),
        max_chars=CONFIG.get("max_chars", None)
    )
elif CONFIG["retriever"] == "cached":
    retriever = CachedRetriever(
        cache_path=CONFIG["dense_scores_path"]
    )
else:
    raise NotImplementedError(f"Retriever type '{CONFIG['retriever']}' not recognized.")

In [ ]:
# === RUN VALIDATION ===
from pathlib import Path

# Set the new flag to True so it returns the DataFrame too!
recall, df_val = evaluate_retriever(retriever, val_data, k=CONFIG["top_k"], return_predictions=True)
print(f"\nValidation Recall@{CONFIG['top_k']}: {recall:.4f}")

In [ ]:
# df_val is the DataFrame returned by evaluate_retriever
cal_metrics = calculate_calibration_metrics(df_val)

In [ ]:
# === SAVE VALIDATION PREDICTIONS ===
from pathlib import Path
from src.evaluation.naming import generate_experiment_name
output_dir = Path("../outputs/validation")
output_dir.mkdir(parents=True, exist_ok=True)

# Generate name dynamically from CONFIG and Environment Variables!
exp_name = generate_experiment_name(CONFIG)
val_name = f"val_{exp_name}.csv" 
output_path = output_dir / val_name
df_val.to_csv(output_path, index=False)
print(f"\n✅ Successfully saved validation predictions to: {output_path.name}")